In [ ]:
!pip install -q --upgrade wandb

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm
import wandb
import random
from itertools import product

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_api")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb_api")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
config = {
    "data_dir": Path("/kaggle/input/competitions/jaguar-re-id"),
    "checkpoint_dir": Path("checkpoints"),
    "megadescriptor_model": "hf-hub:BVRA/MegaDescriptor-L-384",
    "input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "batch_size": 32,
    "dropout": 0.3,
    "val_split": 0.2,
    "seed": SEED,
}

config["checkpoint_dir"].mkdir(exist_ok=True)

# Load data — same split as all previous experiments
train_df = pd.read_csv(config["data_dir"] / "train.csv")
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['ground_truth'])
num_classes = len(label_encoder.classes_)

train_data, val_data = train_test_split(
    train_df,
    test_size=config["val_split"],
    random_state=config["seed"],
    stratify=train_df['ground_truth']
)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Classes: {num_classes}")

In [ ]:
# define model
class EmbeddingProjection(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=256, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim),
        )

    def forward(self, x):
        return self.network(x)

    def get_embeddings(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)


# preprocess
preprocess = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_embeddings(model, image_paths, batch_size=32, desc="Extracting"):
    model.eval()
    embeddings = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc=desc):
        batch_paths = image_paths[i:i+batch_size]
        tensors = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                tensors.append(preprocess(img))
            except:
                tensors.append(torch.zeros(3, config["input_size"],
                                           config["input_size"]))
        batch = torch.stack(tensors).to(device)
        embeddings.append(model(batch).cpu().numpy())
    return np.vstack(embeddings)


# load MegaDescriptor
print("Loading MegaDescriptor...")
megadescriptor = timm.create_model(
    config["megadescriptor_model"], pretrained=True)
megadescriptor.eval()
megadescriptor.to(device)

with torch.no_grad():
    dummy = torch.randn(1, 3, config["input_size"],
                        config["input_size"]).to(device)
    megadescriptor_dim = megadescriptor(dummy).shape[1]

print(f"MegaDescriptor loaded | Dim: {megadescriptor_dim}")

# extract and cache embeddings
cache_dir = Path("embeddings")
cache_dir.mkdir(exist_ok=True)

val_cache = cache_dir / "val_embeddings.npz"
test_cache = cache_dir / "test_embeddings.npz"

# val embeddings
if val_cache.exists():
    val_embeddings = np.load(val_cache)["embeddings"]
    print(f"Loaded val embeddings: {val_embeddings.shape}")
else:
    print("Extracting val embeddings...")
    val_paths = [config["data_dir"] / "train/train" / f
                 for f in val_data['filename'].values]
    val_embeddings = extract_embeddings(megadescriptor, val_paths,
                                        desc="Val")
    np.savez_compressed(val_cache, embeddings=val_embeddings)
    print(f"Saved val embeddings: {val_embeddings.shape}")

# test embeddings
test_pairs_df = pd.read_csv(config["data_dir"] / "test.csv")
test_images = sorted(list(
    set(test_pairs_df['query_image'].unique()) |
    set(test_pairs_df['gallery_image'].unique())
))

if test_cache.exists():
    test_mega_embeddings = np.load(test_cache)["embeddings"]
    print(f"Loaded test embeddings: {test_mega_embeddings.shape}")
else:
    print("Extracting test embeddings...")
    test_paths = [config["data_dir"] / "test/test" / f
                  for f in test_images]
    test_mega_embeddings = extract_embeddings(megadescriptor, test_paths,
                                              desc="Test")
    np.savez_compressed(test_cache, embeddings=test_mega_embeddings)
    print(f"Saved test embeddings: {test_mega_embeddings.shape}")

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class CombinedLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.arcface_weight = nn.Parameter(
            torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.arcface_weight)
        self.margin = 0.5
        self.scale = 64.0

    def arcface_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.arcface_weight, dim=1)
        cosine = F.linear(embeddings, weight).clamp(-1+1e-6, 1-1e-6)
        theta = torch.acos(cosine)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.scale * torch.cos(theta + one_hot * self.margin)
        return F.cross_entropy(output, labels)

    def triplet_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        dist = torch.cdist(embeddings, embeddings, p=2)
        pos_mask = labels.unsqueeze(0) == labels.unsqueeze(1)
        neg_mask = ~pos_mask
        pos_mask.fill_diagonal_(False)
        hardest_pos = (dist * pos_mask.float()).max(dim=1)[0]
        hardest_neg = (dist + 1e6 * (~neg_mask).float()).min(dim=1)[0]
        return F.relu(hardest_pos - hardest_neg + 0.3).mean()

    def forward(self, embeddings, labels):
        return (self.alpha * self.arcface_loss(embeddings, labels) +
                (1 - self.alpha) * self.triplet_loss(embeddings, labels))


# extract train embeddings
train_cache = cache_dir / "train_embeddings.npz"
if train_cache.exists():
    train_embeddings = np.load(train_cache)["embeddings"]
    print(f"Loaded train embeddings: {train_embeddings.shape}")
else:
    print("Extracting train embeddings...")
    train_paths = [config["data_dir"] / "train/train" / f
                   for f in train_data['filename'].values]
    train_embeddings = extract_embeddings(megadescriptor, train_paths,
                                          desc="Train")
    np.savez_compressed(train_cache, embeddings=train_embeddings)
    print(f"Saved train embeddings: {train_embeddings.shape}")

# train model
torch.manual_seed(SEED)
model = EmbeddingProjection(
    input_dim=megadescriptor_dim,
    hidden_dim=config["hidden_dim"],
    output_dim=config["embedding_dim"],
    dropout=config["dropout"]
).to(device)

loss_fn = CombinedLoss(config["embedding_dim"], num_classes).to(device)

train_dataset = EmbeddingDataset(
    train_embeddings, train_data['label_encoded'].values)
train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True, num_workers=0)

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=1e-4, weight_decay=1e-4)

print("Training best model (25 epochs)...")
for epoch in range(25):
    model.train()
    loss_fn.train()
    total_loss = 0
    for embeddings, labels in tqdm(train_loader,
                                   desc=f"Epoch {epoch+1}",
                                   leave=False):
        embeddings, labels = embeddings.to(device), labels.to(device)
        optimizer.zero_grad()
        projected = model(embeddings)
        loss = loss_fn(projected, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:2d} | Loss: {total_loss/len(train_loader):.4f}")

# Get projected embeddings for val and test
model.eval()
with torch.no_grad():
    val_tensor = torch.FloatTensor(val_embeddings).to(device)
    val_projected = model.get_embeddings(val_tensor).cpu().numpy()

    test_tensor = torch.FloatTensor(test_mega_embeddings).to(device)
    test_projected = model.get_embeddings(test_tensor).cpu().numpy()

print(f"Val projected: {val_projected.shape}")
print(f"Test projected: {test_projected.shape}")
print("Model ready")

In [ ]:
def k_reciprocal_reranking(query_features, gallery_features, 
                            k1=20, k2=6, lambda_value=0.3):
    """
    Re-ranks similarity scores using k-reciprocal neighbors.
    Returns final distance matrix (lower = more similar).
    """
    all_features = np.concatenate([query_features, gallery_features], axis=0)
    n_query = query_features.shape[0]
    n_all = all_features.shape[0]

    # compute initial distance matrix
    all_features = all_features / np.linalg.norm(
        all_features, axis=1, keepdims=True)
    dist = 1 - all_features @ all_features.T
    dist = np.maximum(dist, 0)

    # find k-reciprocal neighbors for each sample
    k_reciprocal_neighbors = []
    for i in range(n_all):
        forward_k = np.argsort(dist[i])[:k1+1]
        k_recip = []
        for j in forward_k:
            if i in np.argsort(dist[j])[:k1+1]:
                k_recip.append(j)
        k_reciprocal_neighbors.append(np.array(k_recip))

    # build jaccard distance matrix
    jaccard_dist = np.zeros((n_all, n_all))
    for i in range(n_all):
        for j in range(n_all):
            set_i = set(k_reciprocal_neighbors[i])
            set_j = set(k_reciprocal_neighbors[j])
            intersection = len(set_i & set_j)
            union = len(set_i | set_j)
            jaccard_dist[i][j] = 1 - intersection / (union + 1e-6)

    # final distance
    final_dist = lambda_value * dist + (1 - lambda_value) * jaccard_dist
    
    return final_dist[:n_query, n_query:]


def compute_map_from_dist(dist_matrix, query_labels, gallery_labels):
    """Compute identity-balanced mAP from distance matrix."""
    identity_aps = {}
    for query_idx in range(len(query_labels)):
        query_label = query_labels[query_idx]
        distances = dist_matrix[query_idx]
        is_match = (gallery_labels == query_label).astype(int)

        sorted_indices = np.argsort(distances)  # ascending distance
        sorted_matches = is_match[sorted_indices]

        n_positives = sorted_matches.sum()
        if n_positives == 0:
            continue

        cumsum = np.cumsum(sorted_matches)
        precision_at_k = cumsum / np.arange(1, len(sorted_matches) + 1)
        ap = np.sum(precision_at_k * sorted_matches) / n_positives

        if query_label not in identity_aps:
            identity_aps[query_label] = []
        identity_aps[query_label].append(ap)

    return np.mean([np.mean(aps) for aps in identity_aps.values()])


# baseline val mAP without re-ranking
val_labels = val_data['ground_truth'].values
baseline_dist = 1 - cosine_similarity(val_projected)
np.fill_diagonal(baseline_dist, 1e6)  # exclude self
baseline_map = compute_map_from_dist(baseline_dist, val_labels, val_labels)
print(f"Baseline val mAP (no re-ranking): {baseline_map:.4f}")

print("\nSearching best k1 and lambda parameters...")
print("This may take 10-15 minutes...\n")

# parameter search
k1_values = [10, 15, 20, 25]
lambda_values = [0.2, 0.3, 0.4, 0.5]

search_results = []

for k1, lam in product(k1_values, lambda_values):
    dist_matrix = k_reciprocal_reranking(
        val_projected, val_projected, 
        k1=k1, k2=6, lambda_value=lam)
    np.fill_diagonal(dist_matrix, 1e6)
    val_map = compute_map_from_dist(dist_matrix, val_labels, val_labels)
    search_results.append({
        "k1": k1,
        "lambda": lam,
        "val_map": round(val_map, 4)
    })
    print(f"k1={k1:2d} | lambda={lam} | val_mAP={val_map:.4f}")

# find best parameter
search_df = pd.DataFrame(search_results).sort_values(
    "val_map", ascending=False)
best = search_df.iloc[0]
print(f"\nBest params: k1={best['k1']} | lambda={best['lambda']} "
      f"| val_mAP={best['val_map']}")
print(f"Improvement over baseline: "
      f"{best['val_map'] - baseline_map:+.4f}")

In [ ]:
# apply best re-ranking params to test set
best_k1 = int(best['k1'])
best_lambda = float(best['lambda'])

print(f"Applying re-ranking with k1={best_k1}, lambda={best_lambda}...")

# Re-rank test embeddings
test_dist = k_reciprocal_reranking(
    test_projected, test_projected,
    k1=best_k1, k2=6, lambda_value=best_lambda
)

# convert distance to similarity
test_sim = 1 - test_dist
test_sim = np.clip(test_sim, 0.0, 1.0)

# build image to index mapping
img_to_idx = {fn: idx for idx, fn in enumerate(test_images)}

similarities = []
for _, row in tqdm(test_pairs_df.iterrows(), total=len(test_pairs_df)):
    q_idx = img_to_idx[row['query_image']]
    g_idx = img_to_idx[row['gallery_image']]
    similarities.append(float(test_sim[q_idx, g_idx]))

similarities = np.clip(similarities, 0.0, 1.0)

submission_df = pd.DataFrame({
    'row_id': test_pairs_df['row_id'],
    'similarity': similarities
})

submission_df.to_csv("submission_reranked.csv", index=False)
print(f"Submission saved: {len(submission_df)} rows")

# log to W&B
wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(
    project="jaguar-reid-mishank",
    name="reranking-k1-15-lambda-0.5",
    config={
        "k1": best_k1,
        "k2": 6,
        "lambda": best_lambda,
        "baseline_val_map": baseline_map,
        "reranked_val_map": best['val_map'],
        "improvement": best['val_map'] - baseline_map,
    }
)
wandb.log({
    "baseline_val_map": baseline_map,
    "best_reranked_val_map": best['val_map'],
    "k1": best_k1,
    "lambda": best_lambda,
})

# log full search results
wandb.log({
    "search_results": wandb.Table(dataframe=search_df)
})
wandb.finish()

from IPython.display import FileLink
FileLink('submission_reranked.csv')